# Four-class final evaluation

The selected Decision Tree is retrained from scratch on all eligible four-class training rows. The official test set is used once, after training, with no tuning or threshold changes. Analysis and Backdoor remain outside this model's training scope.

In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report, confusion_matrix

EXPERIMENT = Path.cwd()
PROJECT = EXPERIMENT.parents[2]
DATA = PROJECT / 'aftab' / 'data' / 'processed'
RESULTS = EXPERIMENT / 'results'
PLOTS = RESULTS / 'plots'
MODELS = EXPERIMENT / 'models'
for folder in [RESULTS, PLOTS, MODELS]: folder.mkdir(parents=True, exist_ok=True)
CLASSES = ['Normal', 'Exploits', 'Fuzzers', 'DoS']
TARGET = 'attack_cat'
PREDICTORS = ['dur','proto','service','state','spkts','dpkts','sbytes','dbytes','rate','sttl','dttl','sload','dload','sloss','dloss','sinpkt','dinpkt','sjit','djit','swin','stcpb','dtcpb','dwin','tcprtt','synack','ackdat','smean','dmean','trans_depth','response_body_len','ct_srv_src','ct_state_ttl','ct_dst_ltm','ct_src_dport_ltm','ct_dst_sport_ltm','ct_dst_src_ltm','is_ftp_login','ct_ftp_cmd','ct_flw_http_mthd','ct_src_ltm','ct_srv_dst','is_sm_ips_ports']
assert len(PREDICTORS) == 42 and not set(PREDICTORS) & {'id','label','attack_cat'}
train = pd.read_csv(DATA / 'selected_training.csv')
test = pd.read_csv(DATA / 'selected_testing.csv')
print('Input shapes:', train.shape, test.shape)

## Confirm the selected configuration

The earlier selection used the existing validation partition and did not read the official test data. This final step uses the same unweighted Decision Tree configuration (`random_state=42`) and the same 42 predictors.

In [ ]:
selected_meta = json.loads((MODELS / 'best_four_class_model_metadata.json').read_text())
assert selected_meta['selected_model'] == 'Decision Tree'
assert selected_meta['official_test_used'] is False
assert selected_meta['predictors'] == PREDICTORS
print('Selected model:', selected_meta['selected_model'])
print('Validation macro-F1:', selected_meta['macro_f1'])
print('Official test used during selection:', selected_meta['official_test_used'])

In [ ]:
train4 = train[train[TARGET].isin(CLASSES)].copy()
excluded_counts = test[TARGET].value_counts().drop(labels=CLASSES, errors='ignore').sort_index()
excluded_classes = excluded_counts.index.tolist()
excluded_rows = int(excluded_counts.sum())
test4 = test[test[TARGET].isin(CLASSES)].copy()
print('Training rows used:', len(train4))
print('Test rows used:', len(test4))
print('Excluded test classes:', excluded_classes)
print('Excluded test rows:', excluded_rows)
print('Training class counts:', train4[TARGET].value_counts().reindex(CLASSES, fill_value=0))
print('Test class counts:', test4[TARGET].value_counts().reindex(CLASSES, fill_value=0))
assert set(train4[TARGET]) == set(CLASSES) and set(test4[TARGET]) == set(CLASSES)

## Fit the frozen pipeline

Numeric columns use median imputation. Categorical columns use most-frequent imputation and one-hot encoding. These transformations are fitted only on the full eligible training data.

In [ ]:
X_train = train4[PREDICTORS]
y_train = train4[TARGET]
X_test = test4[PREDICTORS]
y_test = test4[TARGET]
numeric = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical = [c for c in PREDICTORS if c not in numeric]
numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median'))])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True, dtype=np.float32))])
preprocess = ColumnTransformer([('num', numeric_pipe, numeric), ('cat', categorical_pipe, categorical)], remainder='drop')
pipeline = Pipeline([('preprocess', preprocess), ('model', DecisionTreeClassifier(random_state=42))])
start = time.perf_counter()
pipeline.fit(X_train, y_train)
fit_seconds = time.perf_counter() - start
print('Fit seconds:', round(fit_seconds, 3))
print('Numeric predictors:', len(numeric), 'Categorical predictors:', len(categorical))

## Official test evaluation

The frozen pipeline is evaluated once on the retained four-class test rows. Analysis and Backdoor are excluded from this experiment, so this model must not be interpreted as a classifier for those classes.

In [ ]:
pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, pred)
macro_f1 = f1_score(y_test, pred, labels=CLASSES, average='macro')
weighted_f1 = f1_score(y_test, pred, labels=CLASSES, average='weighted')
p, r, f, s = precision_recall_fscore_support(y_test, pred, labels=CLASSES, zero_division=0)
cm = confusion_matrix(y_test, pred, labels=CLASSES)
normal_idx = CLASSES.index('Normal')
normal_fpr = (cm[1:, normal_idx].sum() / cm[1:, :].sum()) if cm[1:, :].sum() else 0.0
attack_recall = r[1:].mean()
report = pd.DataFrame({'class': CLASSES, 'precision': p, 'recall': r, 'f1': f, 'support': s})
metrics = {'accuracy': float(accuracy), 'macro_f1': float(macro_f1), 'weighted_f1': float(weighted_f1), 'attack_recall': float(attack_recall), 'normal_false_positive_rate': float(normal_fpr), 'fit_seconds': float(fit_seconds), 'train_rows': int(len(train4)), 'test_rows_evaluated': int(len(test4)), 'excluded_test_classes': excluded_classes, 'excluded_test_rows': excluded_rows}
print(json.dumps(metrics, indent=2))
print(report)

In [ ]:
# Save new final-evaluation artifacts without changing earlier results.
(RESULTS / 'final_test_metrics.json').write_text(json.dumps(metrics, indent=2))
report.to_csv(RESULTS / 'final_test_classification_report.csv', index=False)
pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(RESULTS / 'final_test_confusion_matrix.csv')
metadata = {'model': 'DecisionTreeClassifier', 'parameters': pipeline.named_steps['model'].get_params(), 'classes': CLASSES, 'predictors': PREDICTORS, 'training_rows': int(len(train4)), 'test_rows_evaluated': int(len(test4)), 'excluded_test_classes': excluded_classes, 'excluded_test_rows': excluded_rows, 'official_test_evaluated': True, 'selection_validation_macro_f1': float(selected_meta['macro_f1']), 'selection_validation_accuracy': float(selected_meta['accuracy']), 'selection_validation_normal_fpr': float(selected_meta['normal_false_positive_rate']), 'note': 'Analysis and Backdoor were excluded from training and evaluation.'}
(RESULTS / 'final_model_metadata.json').write_text(json.dumps(metadata, indent=2, default=str))
joblib.dump(pipeline, MODELS / 'final_four_class_model.joblib')
print('Saved final pipeline and test artifacts.')

In [ ]:
# Raw and normalized confusion matrices.
for matrix, name, fmt in [(cm, 'final_test_confusion.png', 'd'), (cm / cm.sum(axis=1, keepdims=True), 'final_test_confusion_normalized.png', '.2f')]:
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(matrix, cmap='Blues')
    fig.colorbar(im, ax=ax)
    ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=30, ha='right')
    ax.set_yticks(range(len(CLASSES)), CLASSES)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(name.replace('.png',''))
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j, i, format(matrix[i, j], fmt), ha='center', va='center')
    fig.tight_layout(); fig.savefig(PLOTS / name, dpi=140); plt.close(fig)
# Validation values are read from the prior selection record, not recomputed using the refit model.
old = pd.read_csv(RESULTS / 'model_comparison.csv')
val_row = old[(old['model'] == 'Decision Tree') & (old['phase'] == 'baseline')].iloc[0]
comparison = pd.DataFrame({'metric':['macro_f1','accuracy','normal_false_positive_rate','attack_recall'], 'validation':[float(val_row['macro_f1']), float(val_row['accuracy']), float(val_row['normal_false_positive_rate']), float(val_row['attack_recall'])], 'test':[macro_f1, accuracy, normal_fpr, attack_recall]})
comparison.to_csv(RESULTS / 'final_validation_vs_test.csv', index=False)
fig, ax = plt.subplots(figsize=(7,4)); comparison.set_index('metric')[['validation','test']].plot(kind='bar', ax=ax); ax.set_ylim(0,1); ax.set_ylabel('Score'); ax.set_title('Stored validation versus final test'); fig.tight_layout(); fig.savefig(PLOTS / 'final_validation_vs_test.png', dpi=140); plt.close(fig)
summary = '# Final four-class evaluation\n\n' + f"The frozen Decision Tree was retrained on {len(train4):,} eligible training rows and evaluated once on {len(test4):,} retained test rows.\n\n" + f"Test accuracy: {accuracy:.4f}; macro-F1: {macro_f1:.4f}; weighted-F1: {weighted_f1:.4f}; attack recall: {attack_recall:.4f}; Normal false-positive rate: {normal_fpr:.4f}.\n\n" + f"Excluded test classes: {', '.join(excluded_classes)} ({excluded_rows:,} rows). Analysis and Backdoor were excluded from training, so this model cannot reliably identify them. Validation figures are the stored selection results (macro-F1 {float(val_row['macro_f1']):.4f}) and are not test-tuned."
(RESULTS / 'final_evaluation_summary.md').write_text(summary)
print(comparison)